[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/04_diffusion_models/04_diffusion_models.ipynb)

# 04. Diffusion Models for Multimodal Generation

**This notebook covers:**
- Forward diffusion process
- Linear and cosine noise schedules
- Simple denoising U-Net
- DDPM sampling loop
- Cross-attention text conditioning

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/05_Advanced_Topics/04_diffusion_models"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Forward Diffusion Process

$$q(x_t | x_{t-1}) = \mathcal{N}(\sqrt{1-\beta_t}\, x_{t-1}, \beta_t I)$$

Closed form: $x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$.


In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps)
    alphas_cumprod = torch.cos(((x / T) + s) / (1 + s) * np.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return betas.clamp(1e-4, 0.999)

T = 200
betas_lin = linear_beta_schedule(T)
betas_cos = cosine_beta_schedule(T)
alpha_bar_lin = torch.cumprod(1 - betas_lin, dim=0)
alpha_bar_cos = torch.cumprod(1 - betas_cos, dim=0)

plt.plot(alpha_bar_lin.numpy(), label='linear')
plt.plot(alpha_bar_cos.numpy(), label='cosine')
plt.xlabel('timestep t'); plt.ylabel('ᾱ_t'); plt.legend(); plt.title('Noise schedules')
plt.show()

## 2. Forward Diffusion on Synthetic Image


In [ ]:
x0 = torch.zeros(1, 1, 28, 28)
x0[0, 0, 8:20, 8:20] = 1.0  # bright square

def q_sample(x0, t, alpha_bar, noise=None):
    noise = noise if noise is not None else torch.randn_like(x0)
    a = alpha_bar[t].view(-1, 1, 1, 1)
    return torch.sqrt(a) * x0 + torch.sqrt(1 - a) * noise, noise

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for ax, t in zip(axes, [0, 25, 50, 100, 199]):
    xt, _ = q_sample(x0, t, alpha_bar_cos)
    ax.imshow(xt[0, 0].numpy(), cmap='gray'); ax.set_title(f't={t}'); ax.axis('off')
plt.suptitle('Forward diffusion (cosine schedule)'); plt.show()

## 3. Simple Denoising Network


In [ ]:
class SimpleDenoiser(nn.Module):
    def __init__(self, T):
        super().__init__()
        self.time_emb = nn.Embedding(T, 64)
        self.net = nn.Sequential(
            nn.Conv2d(1 + 64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 3, padding=1),
        )
    def forward(self, x, t):
        B = x.size(0)
        te = self.time_emb(t).view(B, 64, 1, 1).expand(-1, -1, x.size(2), x.size(3))
        return self.net(torch.cat([x, te], dim=1))

denoiser = SimpleDenoiser(T)
xt, noise = q_sample(x0, torch.tensor([50]), alpha_bar_cos)
pred = denoiser(xt, torch.tensor([50]))
print(f"Predicted noise shape {pred.shape}, MSE vs true noise: {F.mse_loss(pred, noise).item():.4f}")

## 4. DDPM Sampling Loop


In [ ]:
@torch.no_grad()
def ddpm_sample(model, alpha_bar, betas, shape, steps=None):
    steps = steps or len(betas)
    x = torch.randn(shape)
    for t in reversed(range(steps)):
        t_batch = torch.full((shape[0],), t, dtype=torch.long)
        eps = model(x, t_batch)
        a = alpha_bar[t]
        a_prev = alpha_bar[t-1] if t > 0 else torch.tensor(1.0)
        beta = betas[t]
        coef1 = 1 / torch.sqrt(1 - beta)
        coef2 = beta / torch.sqrt(1 - alpha_bar[t])
        mean = coef1 * (x - coef2 * eps)
        if t > 0:
            x = mean + torch.sqrt(beta) * torch.randn_like(x)
        else:
            x = mean
    return x

# Untrained model -> noisy sample (demonstrates loop mechanics)
sample = ddpm_sample(denoiser, alpha_bar_cos, betas_cos, (1, 1, 28, 28), steps=50)
plt.imshow(sample[0, 0].numpy(), cmap='gray'); plt.title('DDPM sample (untrained denoiser)'); plt.axis('off'); plt.show()

## 5. Cross-Attention Text Conditioning (Stable Diffusion style)


In [ ]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, context):
        B, N, D = x.shape
        M = context.size(1)
        q = self.q_proj(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(context).view(B, M, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(context).view(B, M, self.n_heads, self.d_k).transpose(1, 2)
        attn = F.softmax(q @ k.transpose(-2, -1) / (self.d_k ** 0.5), dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, N, D)
        return self.out(out), attn

block = CrossAttentionBlock(64, 4)
spatial = torch.randn(2, 16, 64)
text_ctx = torch.randn(2, 6, 64)
out, attn = block(spatial, text_ctx)
print(f"Spatial {spatial.shape} + text {text_ctx.shape} -> {out.shape}, attn {attn.shape}")

## Summary

Implemented forward diffusion, noise schedules, a tiny denoiser, DDPM sampling, and cross-attention conditioning.

**Next:** [05_evaluation_benchmarks](../05_evaluation_benchmarks/05_evaluation_benchmarks.ipynb)
